# Preparación, limpieza y estructuración de datos — Pokémon por generaciones

**Autor:** Brandon Maldonado Mesa · Ficha 3169901

Este cuaderno continúa el diagnóstico de calidad (`data_quality.ipynb`) y aplica técnicas de limpieza, filtrado, agrupación y estructuración con Pandas al proyecto individual.

**Pregunta analítica:** ¿cómo han cambiado las estadísticas base promedio de los Pokémon (HP, ataque, defensa, ataque especial, defensa especial y velocidad) entre generaciones, y cuál de ellas ha tenido el mayor cambio?

**Pregunta orientadora:** ¿cómo preparar y estructurar los datos del proyecto individual para obtener información confiable que permita responder la pregunta analítica?

**Contenido**

1. Limpieza y estandarización (Lección 2)
2. Filtrado, ordenamiento y agrupación (Lección 3)
3. Estructuración y validación de los datos (Lección 4)

## Punto de partida: resultados del diagnóstico

El diagnóstico de calidad (`reports/data_quality.md`) concluyó que el dataset está completo y es consistente, y dejó estas acciones de preparación pendientes, que este cuaderno aborda:

1. Mantener los nulos de `tipo_2` y definir una categoría explícita solo en los análisis por tipo.
2. Listar los registros atípicos en `hp`, `defensa` y `defensa_especial` y decidir su tratamiento.
3. Crear una columna auxiliar `especial` (legendario o mítico) para calcular promedios con y sin estos casos.
4. Incluir el número de Pokémon por generación en las tablas de resultados.
5. Documentar las limitaciones: una sola fuente, una sola forma por Pokémon y estadísticas vigentes.

## 0. Preparación del entorno

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw/pokemon_pokeapi.csv')
SALIDA = Path('../data/processed')
SALIDA.mkdir(parents=True, exist_ok=True)

ESTADISTICAS = ['hp', 'ataque', 'defensa', 'ataque_especial', 'defensa_especial', 'velocidad']
TIPOS_VALIDOS = ['normal', 'fire', 'water', 'electric', 'grass', 'ice', 'fighting', 'poison',
                 'ground', 'flying', 'psychic', 'bug', 'rock', 'ghost', 'dragon', 'dark',
                 'steel', 'fairy']
print(pd.__version__)

3.1.0


# Lección 2: limpieza y estandarización

## 1.1 Carga y estado inicial

In [2]:
pokemon_crudo = pd.read_csv(RUTA)
print(pokemon_crudo.shape)
pokemon_crudo.head(3)

(1025, 14)


,id,nombre,generacion,tipo_1,tipo_2,legendario,mitico,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,total
0,1,bulbasaur,1,grass,poison,False,False,45,49,49,65,65,45,318
1,2,ivysaur,1,grass,poison,False,False,60,62,63,80,80,60,405
2,3,venusaur,1,grass,poison,False,False,80,82,83,100,100,80,525


## 1.2 Duplicados

In [3]:
print('Filas duplicadas:', pokemon_crudo.duplicated().sum())
print('IDs repetidos:', pokemon_crudo['id'].duplicated().sum())
print('Nombres repetidos:', pokemon_crudo['nombre'].duplicated().sum())

# Duplicados implícitos: mismo nombre después de estandarizar
nombres_normalizados = pokemon_crudo['nombre'].str.strip().str.lower()
print('Nombres repetidos tras normalizar:', nombres_normalizados.duplicated().sum())

Filas duplicadas: 0
IDs repetidos: 0
Nombres repetidos: 0
Nombres repetidos tras normalizar: 0


**Resultado:** no hay filas, IDs ni nombres duplicados, ni siquiera después de normalizar el texto. **Decisión:** el paso de eliminación de duplicados queda en el pipeline como protección, pero no elimina ninguna fila.

## 1.3 Valores faltantes

In [4]:
nulos = pokemon_crudo.isna().sum()
print(nulos[nulos > 0])
print('tipo_1 sin nulos:', pokemon_crudo['tipo_1'].notna().all())

un_solo_tipo = pokemon_crudo['tipo_2'].isna()
print('Pokémon de un solo tipo:', un_solo_tipo.sum(), '| de dos tipos:', (~un_solo_tipo).sum())

tipo_2    499
dtype: int64
tipo_1 sin nulos: True
Pokémon de un solo tipo: 499 | de dos tipos: 526


**Interpretación:** los 499 nulos de `tipo_2` corresponden a Pokémon de un solo tipo (`tipo_1` nunca es nulo y 499 + 526 = 1025). Es una ausencia con significado, no un error de captura.

**Decisión: conservar el nulo.** Rellenarlo con un texto cambiaría el significado de la columna; si se analiza por tipo secundario, se usará una categoría explícita como `'ninguno'` solo en una copia.

## 1.4 Texto y categorías

In [5]:
for columna in ['nombre', 'tipo_1', 'tipo_2']:
    s = pokemon_crudo[columna].dropna()
    print(f'{columna:7s}',
          '| espacios sobrantes:', (s != s.str.strip()).sum(),
          '| espacios dobles:', s.str.contains('  ', regex=False).sum(),
          '| con mayúsculas:', (s != s.str.lower()).sum())

tipos_usados = set(pokemon_crudo['tipo_1'].dropna()) | set(pokemon_crudo['tipo_2'].dropna())
print('Tipos distintos:', len(tipos_usados))
print('Tipos fuera de la lista oficial:', sorted(tipos_usados - set(TIPOS_VALIDOS)))
print('Nombres con guion (formas o nombres compuestos):', pokemon_crudo['nombre'].str.contains('-').sum())

nombre  | espacios sobrantes: 0 | espacios dobles: 0 | con mayúsculas: 0
tipo_1  | espacios sobrantes: 0 | espacios dobles: 0 | con mayúsculas: 0
tipo_2  | espacios sobrantes: 0 | espacios dobles: 0 | con mayúsculas: 0
Tipos distintos: 18
Tipos fuera de la lista oficial: []
Nombres con guion (formas o nombres compuestos): 76


**Interpretación:** las columnas de texto ya están estandarizadas (minúsculas, sin espacios sobrantes) y los 18 tipos coinciden con la lista oficial. De los 76 nombres con guion, algunos son nombres compuestos (`ho-oh`, `mr-mime`) y otros llevan el sufijo de la forma por defecto del Pokémon (`deoxys-normal`, `pyroar-male`). Se conservan tal como vienen de la API, porque el análisis usa `id` como identificador.

**Decisión:** la función `estandarizar_texto` queda en el pipeline como protección para datos nuevos, sin cambiar valores en este dataset.

## 1.5 Tipos de dato

In [6]:
print(pokemon_crudo.dtypes)
print('Memoria:', round(pokemon_crudo.memory_usage(deep=True).sum() / 1e3, 1), 'KB')

id                  int64
nombre                str
generacion          int64
tipo_1                str
tipo_2                str
legendario           bool
mitico               bool
hp                  int64
ataque              int64
defensa             int64
ataque_especial     int64
defensa_especial    int64
velocidad           int64
total               int64
dtype: object
Memoria: 234.7 KB


**Interpretación:** los tipos coinciden con el significado de cada columna, pero son más grandes de lo necesario (`int64` para valores que no pasan de 255). Se reducirán a `int16` e `int8`, y `tipo_1` y `tipo_2` pasarán a `category`.

## 1.6 Funciones de limpieza y pipeline con informe de cambios

Cada paso es una función que recibe un DataFrame y devuelve otro, y se encadenan con `pipe`. La función `anotar` toma una "foto" de la tabla después de cada paso, sin modificarla, y `validar_datos` comprueba con `assert` las reglas que deben cumplir los datos limpios.

In [7]:
registro = []

def anotar(df, paso):
    registro.append({
        'paso': paso,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'memoria_kb': round(df.memory_usage(deep=True).sum() / 1e3, 1),
    })
    return df

def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def estandarizar_texto(df):
    df = df.copy()
    for columna in ['nombre', 'tipo_1', 'tipo_2']:
        df[columna] = df[columna].str.strip().str.lower()
    return df

def ajustar_tipos(df):
    tipos = {'id': 'int16', 'generacion': 'int8', 'total': 'int16',
             'tipo_1': 'category', 'tipo_2': 'category'}
    tipos.update({estadistica: 'int16' for estadistica in ESTADISTICAS})
    return df.astype(tipos)

def agregar_especial(df):
    df = df.copy()
    df['especial'] = df['legendario'] | df['mitico']
    return df

def validar_datos(df):
    assert len(df) == 1025, 'número de filas inesperado'
    assert df['id'].tolist() == list(range(1, 1026)), 'IDs no consecutivos'
    assert df['nombre'].is_unique, 'nombres repetidos'
    assert df['generacion'].between(1, 9).all(), 'generación fuera de rango'
    assert df[ESTADISTICAS].gt(0).all().all(), 'estadística menor o igual a 0'
    assert df[ESTADISTICAS].le(255).all().all(), 'estadística mayor a 255'
    assert (df[ESTADISTICAS].sum(axis=1) == df['total']).all(), 'total distinto de la suma'
    assert not (df['legendario'] & df['mitico']).any(), 'legendario y mítico a la vez'
    assert df['tipo_1'].notna().all(), 'tipo_1 con nulos'
    assert df['tipo_1'].isin(TIPOS_VALIDOS).all(), 'tipo_1 inválido'
    assert (df['tipo_2'].isna() | df['tipo_2'].isin(TIPOS_VALIDOS)).all(), 'tipo_2 inválido'
    assert (df['tipo_2'].dropna() != df.loc[df['tipo_2'].notna(), 'tipo_1']).all(), 'tipo_2 igual a tipo_1'
    return df

In [8]:
pokemon = (pokemon_crudo
    .pipe(anotar, '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, '1 sin duplicados')
    .pipe(estandarizar_texto)
    .pipe(anotar, '2 texto estandarizado')
    .pipe(ajustar_tipos)
    .pipe(anotar, '3 tipos')
    .pipe(agregar_especial)
    .pipe(anotar, '4 columna especial')
    .pipe(validar_datos))

informe_limpieza = pd.DataFrame(registro)
informe_limpieza

,paso,filas,columnas,nulos,memoria_kb
0,0 crudo,1025,14,499,234.7
1,1 sin duplicados,1025,14,499,234.7
2,2 texto estandarizado,1025,14,499,234.7
3,3 tipos,1025,14,499,82.0
4,4 columna especial,1025,15,499,83.0


**Interpretación:** el pipeline no eliminó filas ni cambió valores. Ajustó los tipos de dato (la memoria baja de 117,2 a 38,6 KB), agregó la columna `especial` (legendario o mítico) y superó todas las validaciones. Los 499 nulos de `tipo_2` se mantienen a propósito.

## 1.7 Estado inicial y final de los datos

In [9]:
def metricas_calidad(df):
    estadisticas = df[ESTADISTICAS]
    tipos_invalidos = (~df['tipo_1'].isin(TIPOS_VALIDOS)).sum() + \
                      (df['tipo_2'].notna() & ~df['tipo_2'].isin(TIPOS_VALIDOS)).sum()
    return {
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos_totales': int(df.isna().sum().sum()),
        'filas_duplicadas': int(df.duplicated().sum()),
        'ids_repetidos': int(df['id'].duplicated().sum()),
        'total_distinto_de_suma': int((estadisticas.sum(axis=1) != df['total']).sum()),
        'estadisticas_fuera_de_rango': int(((estadisticas <= 0) | (estadisticas > 255)).sum().sum()),
        'generacion_fuera_de_rango': int((~df['generacion'].between(1, 9)).sum()),
        'tipos_invalidos': int(tipos_invalidos),
    }

comparacion_calidad = pd.DataFrame({'antes': metricas_calidad(pokemon_crudo),
                                    'despues': metricas_calidad(pokemon)}).astype(object)
comparacion_calidad.loc['memoria_kb'] = [informe_limpieza['memoria_kb'].iloc[0],
                                         informe_limpieza['memoria_kb'].iloc[-1]]
comparacion_calidad

,antes,despues
filas,1025,1025
columnas,14,15
nulos_totales,499,499
filas_duplicadas,0,0
ids_repetidos,0,0
total_distinto_de_suma,0,0
estadisticas_fuera_de_rango,0,0
generacion_fuera_de_rango,0,0
tipos_invalidos,0,0
memoria_kb,234.7,83.0


**Comparación antes y después:** las verificaciones de calidad dan cero casos antes y después (el dataset de origen ya era consistente, como concluyó el diagnóstico). Los cambios reales son la reducción de memoria (117,2 a 39,6 KB), los tipos de dato ajustados y la columna nueva `especial`.

In [10]:
pd.DataFrame({'antes': pokemon_crudo.dtypes.astype(str),
              'despues': pokemon.dtypes.astype(str)}).fillna('(columna nueva)')

,antes,despues
ataque,int64,int16
ataque_especial,int64,int16
defensa,int64,int16
defensa_especial,int64,int16
especial,(columna nueva),bool
generacion,int64,int8
hp,int64,int16
id,int64,int16
legendario,bool,bool
mitico,bool,bool
